# Encontrando Direcionadores de Demanda com PROC GLMSELECT: Seleção Stepwise, LASSO e Forward Validada

## Resumo Executivo

Uma equipe de análise de varejo usa o **PROC GLMSELECT** para descobrir quais alavancas de marketing e preço realmente movem as vendas semanais de unidades, separando os verdadeiros direcionadores de demanda do ruído. A seleção stepwise pontuada pelo SBC, o LASSO com validação cruzada de 5 dobras e uma busca forward validada por holdout recuperam **o mesmo conjunto de direcionadores verdadeiros** — preço próprio, preço do concorrente, gasto em publicidade, volume de e-mails, promoções, feriados, a região Northeast e o canal Online — e cada uma das quatro iscas plantadas (`temp_f`, `noise1`, `noise2`, `noise3`) é automaticamente descartada.

Os métodos também concordam de perto quanto às magnitudes: cada um estima um efeito de preço próprio perto de **-28 unidades por dólar** e um efeito de preço do concorrente perto de **+14**, os sinais de substituição embutidos na equação geradora dos dados. A única discordância está na margem — o LASSO com validação cruzada mantém adicionalmente um pequeno contraste `Region=Midwest`, estatisticamente insignificante (estimativa -8.3 com erro padrão de 8.3), que tanto a seleção stepwise quanto a forward descartam. Uma lista de direcionadores que sobrevive a três filosofias de seleção diferentes é muito mais confiável do que uma ajustada a uma única regra.

## Fontes de Dados

Todos os dados neste notebook são **sintéticos** e gerados diretamente no código (sem arquivos externos, seed `20260531`). Eles simulam uma temporada de painéis de demanda loja-semana para um varejista de bens de consumo.

| Dataset | Linhas | Granularidade | Variáveis-chave |
|---------|------|-------|---------------|
| `demand` | 100 | Loja-semana | `units` (resposta: unidades vendidas na semana); `price`, `competitor` (preço próprio e do concorrente na prateleira); `adspend`, `email` (pressão de mídia); `promo`, `holiday` (indicadores de evento); `region`, `channel` (efeitos CLASS); `temp_f`, `noise1`-`noise3` (preditores isca/irrelevantes) |

`units` é construída a partir de uma equação linear conhecida, de modo que o conjunto "correto" de direcionadores seja verificável; `temp_f` e as três colunas `noise` não carregam sinal real e existem para testar se cada método de seleção as descarta.

# Encontrando Direcionadores de Demanda com PROC GLMSELECT

Um gerente de categoria tem dezenas de explicadores candidatos para as vendas semanais: preço próprio, preço do concorrente, gasto em publicidade, volume de e-mails, promoções, feriados, região da loja, canal de vendas, até o clima. Jogar todos eles em uma única regressão convida ao overfitting e a coeficientes instáveis. O **PROC GLMSELECT** automatiza a busca por um modelo parcimonioso, suportando seleção stepwise, LASSO, elastic net e least-angle, com validação cruzada e particionamento de holdout embutidos.

Neste notebook nós:

1. Geramos um painel de demanda loja-semana sintético e realista com um conjunto *conhecido* de direcionadores verdadeiros (além de variáveis isca deliberadas).
2. Executamos a **seleção stepwise** pontuada pelo SBC.
3. Executamos o **LASSO** com validação cruzada de 5 dobras.
4. Executamos a **seleção forward validada em um holdout de 30%**.

Um bom método de seleção deve recuperar os direcionadores reais e descartar o ruído — vamos ver.

## 1. Gerar um painel de demanda sintético

A resposta `units` é construída a partir de uma equação linear explícita, então conhecemos a verdade fundamental: preço e preço do concorrente, gasto em publicidade, volume de e-mails, os indicadores de promo e feriado, além dos efeitos de região e canal, todos importam. As variáveis `temp_f`, `noise1`, `noise2` e `noise3` são iscas puras, sem relação real com as vendas. Uma seed em `call streaminit` torna os dados reproduzíveis.

In [1]:
/* ---------------------------------------------------------------
   Painel sintético de demanda loja-semana para um varejista de bens de consumo.
   'units' segue uma equação CONHECIDA; temp_f e noise1-3 são iscas (decoys).
   --------------------------------------------------------------- */
DADOS demand;
    CHAMAR streaminit(20260531);
    COMPRIMENTO region $9 channel $8 promo $3;
    FAZER store_week = 1 ATÉ 100;
        /* Composição da região */
        u1 = rand('uniform');
        SE u1 < 0.34 ENTÃO region = 'Northeast';
        SENÃO SE u1 < 0.67 ENTÃO region = 'Midwest';
        SENÃO region = 'South';

        /* Canal de vendas */
        SE rand('uniform') < 0.45 ENTÃO channel = 'Store';
        SENÃO channel = 'Online';

        /* Direcionadores de preço e mídia */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Indicadores de evento e uma isca climática irrelevante */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        SE rand('uniform') < 0.40 ENTÃO promo = 'Yes';
        SENÃO promo = 'No';

        /* Preditores de ruído puro (sem efeito real) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Unidades semanais vendidas a partir de uma equação estrutural conhecida */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        SE units < 0 ENTÃO units = 0;
        SAÍDA;
    FIM;
    MANTER region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
EXECUTAR;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Perfilar os dados

Antes de modelar, confirme que a resposta e os principais candidatos contínuos estão em escalas sensatas.

In [2]:
PROCEDIMENTO MÉDIAS DADOS=demand n mean std MIN MAX maxdec=1;
    VARIÁVEL units price competitor adspend email;
    RÓTULO units="Unidades vendidas" price="Preço" competitor="Preço do concorrente"
          adspend="Gasto em publicidade" email="Volume de e-mails";
    TÍTULO "Demanda semanal e direcionadores candidatos";
EXECUTAR;

                                      Demanda semanal e direcionadores candidatos                                       

                                                  The MEANS Procedure

 Variable    Label                         N        Mean     Std Dev     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 units       Unidades vendidas           100       874.8       136.3       508.6      1162.3
 price       Preço                       100        14.0         3.4         8.0        19.9
 competitor  Preço do concorrente        100        13.8         3.4         8.1        19.9
 adspend     Gasto em publicidade        100       990.6       726.9        50.0      3358.0
 email       Volume de e-mails           100        45.5        26.4         0.0        99.0
 -------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Seleção stepwise pontuada pelo SBC

A seleção stepwise alternadamente adiciona e remove efeitos, aqui julgados pelo **Critério Bayesiano de Schwarz (SBC)** tanto para o teste de entrada (`select=sbc`) quanto para a escolha final do modelo (`choose=sbc`). O SBC penaliza a complexidade mais fortemente que o AIC, favorecendo modelos mais enxutos.

Principais statements e opções:

- `CLASS region channel promo / param=reference` declara os preditores categóricos com codificação de célula de referência.
- `selection=stepwise(select=sbc choose=sbc)` conduz a busca.
- `details=summary` imprime o resumo passo a passo da seleção; `stb` adiciona coeficientes padronizados para que efeitos em escalas diferentes sejam comparáveis.
- `output out=demand_scored p=predicted r=residual` salva os valores ajustados e os resíduos para pontuação posterior.

Leia o **Stepwise Selection Summary** como o traço da busca: ele começa no modelo completo de 12 efeitos e *remove* efeitos um a um, descartando `noise1`, `noise2`, `temp_f`, o contraste `Region=Midwest` e `noise3`, nessa ordem, porque cada remoção reduz o SBC. O que sobrevive na tabela **Parameter Estimates** é o modelo escolhido.

In [3]:
PROCEDIMENTO glmselect DADOS=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(SELECIONAR=sbc choose=sbc)
          details=summary stb;
    SAÍDA out=demand_scored p=predicted r=residual;
    RÓTULO units="Unidades vendidas" region="Região" channel="Canal" promo="Promoção"
          price="Preço" competitor="Preço do concorrente" adspend="Gasto em publicidade"
          email="Volume de e-mails" temp_f="Temperatura (F)" holiday="Feriado"
          noise1="Ruído 1" noise2="Ruído 2" noise3="Ruído 3";
    TÍTULO "Seleção stepwise dos direcionadores de demanda (SBC)";
EXECUTAR;

                                      Demanda semanal e direcionadores candidatos                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                 Stepwise Selection Summary                                                 

    Step    Action           Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ---------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed          Ruído 1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
     


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO com validação cruzada

O LASSO encolhe os coeficientes em direção a zero, realizando seleção e regularização simultaneamente. Em vez de parar em um critério fixo, deixamos a **validação cruzada de 5 dobras** escolher o ponto no caminho do LASSO com o menor erro fora da amostra.

- `selection=lasso(choose=cv stop=none)` traça o caminho completo do LASSO e seleciona o passo ótimo pela validação cruzada.
- `cvmethod=random(5)` atribui as observações a 5 dobras aleatórias.

O **LASSO Selection Summary** mostra a ordem em que os efeitos entram conforme a penalidade se relaxa: `price` primeiro, depois `adspend`, `competitor`, a região Northeast, `email`, `promo` e `holiday` — todos os sete sinais verdadeiros entram antes de qualquer isca. A validação cruzada então escolhe o passo em que o PRESS fora da amostra é o menor, e a tabela **Parameter Estimates** resultante mantém exatamente os direcionadores genuínos (mais um pequeno termo `Region=Midwest`) enquanto exclui `temp_f`, `noise1`, `noise2` e `noise3`, que só entram bem no final do caminho.

In [4]:
PROCEDIMENTO glmselect DADOS=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv PARAR=none)
          cvmethod=RANDOM(5);
    RÓTULO units="Unidades vendidas" region="Região" channel="Canal" promo="Promoção"
          price="Preço" competitor="Preço do concorrente" adspend="Gasto em publicidade"
          email="Volume de e-mails" temp_f="Temperatura (F)" holiday="Feriado"
          noise1="Ruído 1" noise2="Ruído 2" noise3="Ruído 3";
    TÍTULO "LASSO com validação cruzada de 5 dobras";
EXECUTAR;

                                      Demanda semanal e direcionadores candidatos                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                            LASSO Selection Summary                                                            

    Step    Action                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ---------------------  --


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Seleção forward validada em um holdout

Uma terceira estratégia complementar: construir o modelo por **seleção forward** (os efeitos apenas entram, nunca saem), mas parar onde o desempenho em uma amostra de holdout independente é melhor — protegendo diretamente contra o overfitting.

- `partition fraction(validate=0.30)` reserva aleatoriamente 30% das linhas para validação, deixando 70 observações de treino e 30 de validação.
- `selection=forward(select=aic choose=validate)` adiciona efeitos pelo AIC, mas escolhe o modelo final pelo erro na amostra de validação.

A tabela **Partition Fractions** confirma a divisão 70/30. A seleção forward então entra com efeitos até que o erro de validação pare de melhorar; os oito efeitos na tabela final **Parameter Estimates** são precisamente os direcionadores verdadeiros, com as quatro iscas nunca admitidas. Quando três métodos construídos sobre princípios diferentes chegam aos mesmos direcionadores, o modelo é muito mais provável de ser real do que um artefato de uma única regra de seleção.

In [5]:
PROCEDIMENTO glmselect DADOS=demand seed=20260531;
    CLASSE region channel promo / PARAM=reference;
    MODELO units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(SELECIONAR=aic choose=validate);
    partition FRACTION(validate=0.30);
    RÓTULO units="Unidades vendidas" region="Região" channel="Canal" promo="Promoção"
          price="Preço" competitor="Preço do concorrente" adspend="Gasto em publicidade"
          email="Volume de e-mails" temp_f="Temperatura (F)" holiday="Feriado"
          noise1="Ruído 1" noise2="Ruído 2" noise3="Ruído 3";
    TÍTULO "Seleção forward validada em um holdout de 30%";
EXECUTAR;

                                      Demanda semanal e direcionadores candidatos                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Unidades vendidas


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                           Forward Selection Summary                                                           

    Step    Action                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  --------


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpretando os resultados

As três estratégias de seleção recuperam **o mesmo conjunto de direcionadores de demanda verdadeiros** e descartam todas as iscas. Lendo diretamente das tabelas **Parameter Estimates**:

- **Preço** é o direcionador dominante. Seu coeficiente padronizado no modelo stepwise é **-0.70**, de longe o maior em magnitude, e a inclinação bruta fica entre **-27.8** (stepwise e LASSO) e **-28.6** (forward) unidades por dólar — quase exatamente o -28 com que os dados foram gerados. O **preço do concorrente** move a demanda na direção oposta, em torno de **+14.4** nos três ajustes, o comportamento de substituição que um gerente de categoria espera.
- O **gasto em publicidade** (cerca de +0.062 unidades por dólar) e o **volume de e-mails** (cerca de +1.2 unidades por envio) elevam as vendas, quantificando a resposta de mídia.
- **Promoções** e **feriados** carregam os maiores efeitos de evento: o contraste `Promo=No` fica em torno de **-51 a -57** em relação a uma semana promovida, e o aumento por feriado é de aproximadamente **+66 a +76** unidades. A **região Northeast** (cerca de +49 a +55) e o **canal Online** (cerca de +24 a +32) carregam efeitos de base positivos.
- Criticamente, cada isca — `temp_f`, `noise1`, `noise2`, `noise3` — é **descartada** pela seleção stepwise e forward e é excluída do modelo LASSO escolhido por CV. Cada método recuperou o sinal estrutural e ignorou o ruído.

Os três métodos não são idênticos byte a byte: o stepwise (SBC) e a busca forward validada por holdout chegam aos mesmos oito efeitos, enquanto o LASSO com validação cruzada retém adicionalmente um pequeno contraste `Region=Midwest` (-8.3, erro padrão 8.3) — uma diferença no piso de ruído, e não uma discordância substantiva. O fato de uma lista de direcionadores sobreviver à seleção stepwise SBC, ao LASSO com validação cruzada e à seleção forward validada por holdout é a verdadeira conclusão: um achado robusto a três filosofias de seleção diferentes é muito mais confiável do que um ajustado a um único critério. Com `OUTPUT OUT=demand_scored` capturando os valores ajustados e os resíduos, o mesmo fluxo de trabalho se estende naturalmente para pontuar o calendário de preços e promoções planejado para o próximo trimestre.